# Analisis analitico usando LLM

Para este analisís se hizo uso de un LLM para poder analizar las transcripciones y poder obtener metricas mas complejas y abstractas.


### Análisis de las variables etiquetadas con LLM

El cuaderno determinístico (`base_analitica_deterministica.ipynb`) mide **cómo** habla cada
agente: tiempos, silencios y patrones de texto. Eso no alcanza para responder la pregunta
central de la prueba --- *¿quién es más efectivo?* --- porque el resultado de una negociación
es semántico: hay que leer la conversación y juzgarla.

Este cuaderno cierra esa brecha. Toma las etiquetas que produjo el LLM sobre los turnos
exportados y las convierte en cinco variables comparables entre humanos e IA:

| Variable | Qué captura |
|---|---|
| `resultado_final` | desenlace de la negociación (compromiso firme, vago, rechazo, recontacto, sin conclusión) |
| `tipo_objecion_principal` | la resistencia que puso el deudor |
| `concesion_ante_objecion` | si el agente respondió con algo concreto o repitió el guion |
| `cierre_operativo` | si quedó claro cómo, cuándo y dónde pagar |
| `tono_agente` | tono predominante del agente |

**Entrada:** `Transcriptions procesadas/Transcriptions_conTurnos/json_LLM/{humano,ai}-analisis.json`
**Salida:** `Base analitica/base_llm.csv` (una fila por llamada) y `contrastes_llm.csv` (los tests).

## Prompt usado


Eres un analista que revisa transcripciones de llamadas de cobranza (call center) en
español. Cada llamada ya tiene los turnos etiquetados como AGENTE o DEUDOR (y, en pocos
casos, INCIERTO cuando no se pudo determinar el rol con certeza).

Para cada llamada, evalúa 5 dimensiones:

1. "resultado_final": el desenlace de la negociación. Uno de:
   - "compromiso_firme": el deudor acepta pagar con fecha Y monto concretos.
   - "compromiso_vago": acepta la idea de pagar pero sin fecha y/o monto claros.
   - "rechazo": se niega a pagar o a negociar.
   - "recontacto": queda pendiente de una decisión o llamada futura, sin cerrar nada hoy.
   - "sin_conclusion": la llamada termina sin que se pueda ubicar en ninguna de las
     anteriores (se corta, queda inconclusa, etc.).

2. "tipo_objecion_principal": la objeción más relevante que puso el deudor (si puso varias,
   elige la que más peso tuvo en la conversación). Uno de:
   - "economica": no tiene dinero o dificultades de pago.
   - "desconocimiento_desconfianza": no reconoce la deuda, duda de que sea legítima, pide
     verificar quién llama.
   - "logistica": no puede atender en este momento, pide que lo llamen después.
   - "otra": una objeción real que no cabe en las anteriores.
   - "sin_objecion": el deudor no puso resistencia real.

3. "concesion_ante_objecion": si hubo objeción, ¿el agente respondió con algo concreto y
   distinto a lo ya ofrecido (nueva fecha, nuevo monto, plan de cuotas, descuento adicional),
   o repitió lo mismo / solo mostró comprensión sin cambiar nada?
   Responde "si", "no", o "no_aplica" (si tipo_objecion_principal es "sin_objecion").

4. "cierre_operativo": al terminar la llamada, ¿quedó claro CÓMO, CUÁNDO y/o DÓNDE debe pagar
   el deudor (canal de pago, fecha, monto)? No cuenta un "le enviaremos la información" sin
   ningún detalle concreto. Responde "si" o "no".

5. "tono_agente": el tono predominante del agente durante la llamada. Uno de "cordial",
   "neutro", "agresivo".

Para cada evaluación, incluye una "evidencia" MUY corta (máx. 10 palabras) citando o
resumiendo lo que sustenta tu decisión, y una "confianza" ("alta", "media", "baja").

Responde ÚNICAMENTE con un JSON válido (sin texto antes ni después, sin markdown, sin
```json), con esta estructura exacta:

{
  "resultados": [
    {
      "short_id": "audioh1-0445c357-2spk",
      "resultado_final": "compromiso_firme",
      "resultado_evidencia": "acuerda 6 cuotas de $133.300",
      "resultado_confianza": "alta",
      "tipo_objecion_principal": "economica",
      "objecion_evidencia": "dice que no tiene el dinero ahora",
      "concesion_ante_objecion": "si",
      "concesion_evidencia": "el agente ofrece plan de 6 cuotas",
      "cierre_operativo": "si",
      "cierre_evidencia": "indica pagar con número de convenio",
      "tono_agente": "cordial"
    }
  ]
}

Debe haber un objeto en "resultados" por cada llamada que te paso, ni más ni menos.

Aquí están las llamadas:

--- INICIO DE DATOS ---
[PEGAR BLOQUE]
--- FIN DE DATOS ---



## Cargar y validar

Dos controles antes de analizar nada, porque el insumo lo produjo un modelo y no un proceso
determinístico:

1. **Cobertura:** cuántas llamadas quedaron etiquetadas de las 97 del corpus.
2. **Catálogo:** que el LLM no haya inventado categorías fuera de las que pedía el prompt.
   Si aparece un valor nuevo, hay que verlo antes de contarlo, no después.

In [6]:
import json
import csv
from collections import Counter
from math import comb
from pathlib import Path

REPO_ROOT = Path(".").resolve()
if not (REPO_ROOT / "Transcriptions procesadas").exists():
    REPO_ROOT = REPO_ROOT.parent  # por si se corre desde /Scripts

JSON_LLM = REPO_ROOT / "Transcriptions procesadas" / "Transcriptions_conTurnos" / "json_LLM"
OUT_DIR = REPO_ROOT / "Base analitica"
OUT_DIR.mkdir(exist_ok=True)

# Catálogo de valores válidos: es el mismo que declara el prompt de arriba
CATEGORIAS = {
    "resultado_final": ["compromiso_firme", "compromiso_vago", "rechazo", "recontacto", "sin_conclusion"],
    "tipo_objecion_principal": ["economica", "desconocimiento_desconfianza", "logistica", "otra", "sin_objecion"],
    "concesion_ante_objecion": ["si", "no", "no_aplica"],
    "cierre_operativo": ["si", "no"],
    "tono_agente": ["cordial", "neutro", "agresivo"],
    "resultado_confianza": ["alta", "media", "baja"],
}

BASE = [{"grupo": grupo, **r}
        for grupo, archivo in [("humano", "humano-analisis.json"), ("ia", "ai-analisis.json")]
        for r in json.loads((JSON_LLM / archivo).read_text(encoding="utf-8"))["resultados"]]

H = [r for r in BASE if r["grupo"] == "humano"]
IA = [r for r in BASE if r["grupo"] == "ia"]
print(f"Etiquetadas: {len(BASE)}/97   (humano {len(H)}/48, IA {len(IA)}/49)")

fuera = {var: {r["short_id"]: r.get(var) for r in BASE if r.get(var) not in validos}
         for var, validos in CATEGORIAS.items()}
fuera = {v: d for v, d in fuera.items() if d}
print("Valores fuera del catálogo:", fuera or "ninguno")

Etiquetadas: 97/97   (humano 48/48, IA 49/49)
Valores fuera del catálogo: ninguno


## Distribuciones

Las cinco variables son categóricas, así que la estadística descriptiva es la proporción de
llamadas en cada categoría. Se reporta el conteo y el porcentaje sobre las llamadas
**etiquetadas de cada grupo** (no sobre las 97), porque la cobertura es distinta entre grupos.

In [7]:
def distribucion(var):
    ch, ci = Counter(r[var] for r in H), Counter(r[var] for r in IA)
    print(f"\n{var}")
    print(f"  {'categoría':<32}{'humano':>13}{'IA':>13}")
    for cat in CATEGORIAS[var]:
        h, i = ch.get(cat, 0), ci.get(cat, 0)
        print(f"  {cat:<32}{f'{h} ({100 * h / len(H):.0f}%)':>13}{f'{i} ({100 * i / len(IA):.0f}%)':>13}")


for var in CATEGORIAS:
    distribucion(var)


resultado_final
  categoría                              humano           IA
  compromiso_firme                     27 (56%)     12 (24%)
  compromiso_vago                        4 (8%)       0 (0%)
  rechazo                                3 (6%)      7 (14%)
  recontacto                           12 (25%)      8 (16%)
  sin_conclusion                         2 (4%)     22 (45%)

tipo_objecion_principal
  categoría                              humano           IA
  economica                            19 (40%)     14 (29%)
  desconocimiento_desconfianza          5 (10%)     14 (29%)
  logistica                             7 (15%)      6 (12%)
  otra                                  8 (17%)       3 (6%)
  sin_objecion                          9 (19%)     12 (24%)

concesion_ante_objecion
  categoría                              humano           IA
  si                                   23 (48%)     12 (24%)
  no                                   16 (33%)     25 (51%)
  no_aplica       

## Contrastes

Dos decisiones de método:

**Por qué la prueba exacta de Fisher y no chi-cuadrado.** Con ~45 llamadas por grupo, varias
categorías quedan con muy pocos casos (`compromiso_vago` tiene 3 en humanos y 0 en IA).
Chi-cuadrado es una aproximación que exige celdas suficientemente grandes; Fisher calcula la
probabilidad exacta y es válido con celdas pequeñas.

**Por qué binarizar en vez de contrastar la tabla completa.** Una tabla de 2×5 responde
"¿las distribuciones diferen?", que no es una conclusión accionable. Binarizar en la categoría
que importa ("¿cierra compromiso firme, sí o no?") responde una pregunta concreta y da un
tamaño de efecto directamente comunicable: la diferencia en puntos porcentuales.

`Concede ante objeción` se calcula solo sobre las llamadas **donde hubo objeción** (excluyendo
`no_aplica`): incluir las llamadas sin objeción en el denominador confundiría "no concedió" con
"no tuvo nada que conceder".

In [8]:
def fisher_2x2(a, b, c, d):
    """p-valor bilateral de la prueba exacta de Fisher para la tabla [[a,b],[c,d]].
    Suma la probabilidad hipergeométrica de todas las tablas tan o más extremas
    que la observada, manteniendo fijos los totales marginales."""
    fila1, fila2, col1, total = a + b, c + d, a + c, a + b + c + d
    prob = lambda x: comb(fila1, x) * comb(fila2, col1 - x) / comb(total, col1)
    p_obs = prob(a)
    lo, hi = max(0, col1 - fila2), min(fila1, col1)
    return min(1.0, sum(prob(x) for x in range(lo, hi + 1) if prob(x) <= p_obs * (1 + 1e-9)))


# (nombre, variable, categorías que cuentan como éxito, universo a considerar)
CONTRASTES = [
    ("Cierra compromiso firme",    "resultado_final",          {"compromiso_firme"},              None),
    ("Llamada sin conclusión",     "resultado_final",          {"sin_conclusion"},                None),
    ("Cierre operativo claro",     "cierre_operativo",         {"si"},                            None),
    ("Concede ante objeción",      "concesion_ante_objecion",  {"si"},               {"si", "no"}),
    ("Objeción por desconfianza",  "tipo_objecion_principal",  {"desconocimiento_desconfianza"},  None),
    ("Tono cordial",               "tono_agente",              {"cordial"},                       None),
]

CONTRASTES_OUT = []
print(f"{'contraste':<28}{'humano':>12}{'IA':>12}{'dif pp':>9}{'p':>10}")
print("-" * 72)
for nombre, var, exito, universo in CONTRASTES:
    h = [r for r in H if universo is None or r[var] in universo]
    i = [r for r in IA if universo is None or r[var] in universo]
    a, c = sum(r[var] in exito for r in h), sum(r[var] in exito for r in i)
    b, d = len(h) - a, len(i) - c
    pct_h, pct_i = 100 * a / len(h), 100 * c / len(i)
    p = fisher_2x2(a, b, c, d)
    estrellas = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"{nombre:<28}{f'{a}/{len(h)} {pct_h:.0f}%':>12}{f'{c}/{len(i)} {pct_i:.0f}%':>12}"
          f"{pct_i - pct_h:>+9.0f}{p:>9.4f} {estrellas}")
    CONTRASTES_OUT.append({"contraste": nombre, "variable": var,
                           "humano_exitos": a, "humano_n": len(h), "humano_pct": round(pct_h, 1),
                           "ia_exitos": c, "ia_n": len(i), "ia_pct": round(pct_i, 1),
                           "dif_pp": round(pct_i - pct_h, 1), "p_fisher": round(p, 5)})

print("\n*** p<0.001   ** p<0.01   * p<0.05   ns = no significativo")

contraste                         humano          IA   dif pp         p
------------------------------------------------------------------------
Cierra compromiso firme        27/48 56%   12/49 24%      -32   0.0019 **
Llamada sin conclusión           2/48 4%   22/49 45%      +41   0.0000 ***
Cierre operativo claro         27/48 56%   10/49 20%      -36   0.0004 ***
Concede ante objeción          23/39 59%   12/37 32%      -27   0.0237 *
Objeción por desconfianza       5/48 10%   14/49 29%      +18   0.0391 *
Tono cordial                   44/48 92%   20/49 41%      -51   0.0000 ***

*** p<0.001   ** p<0.01   * p<0.05   ns = no significativo


## Robustez

El LLM reporta su propia confianza por llamada. Si el hallazgo principal solo existiera gracias
a las etiquetas dudosas, no sería creíble. Se repite el contraste central usando únicamente las
llamadas de confianza `alta`.

In [4]:
ha = [r for r in H if r["resultado_confianza"] == "alta"]
ia_alta = [r for r in IA if r["resultado_confianza"] == "alta"]
a = sum(r["resultado_final"] == "compromiso_firme" for r in ha)
c = sum(r["resultado_final"] == "compromiso_firme" for r in ia_alta)

print(f"Confianza alta: humano {len(ha)}/{len(H)}, IA {len(ia_alta)}/{len(IA)}")
print(f"Compromiso firme (solo confianza alta): humano {a}/{len(ha)} ({100 * a / len(ha):.0f}%) "
      f"vs IA {c}/{len(ia_alta)} ({100 * c / len(ia_alta):.0f}%)")
print(f"p = {fisher_2x2(a, len(ha) - a, c, len(ia_alta) - c):.4f}")

Confianza alta: humano 32/48, IA 42/49
Compromiso firme (solo confianza alta): humano 23/32 (72%) vs IA 8/42 (19%)
p = 0.0000


## Guardar

In [5]:
COLS = ["grupo", "short_id"] + list(CATEGORIAS) + [
    "resultado_evidencia", "objecion_evidencia", "concesion_evidencia", "cierre_evidencia"]

for nombre, filas, cols in [("base_llm.csv", BASE, COLS),
                            ("contrastes_llm.csv", CONTRASTES_OUT, list(CONTRASTES_OUT[0]))]:
    with open(OUT_DIR / nombre, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        w.writerows(filas)
    print(f"{nombre}: {len(filas)} filas")

base_llm.csv: 97 filas
contrastes_llm.csv: 6 filas


## Lectura de los resultados

Los cinco contrastes significativos apuntan en la misma dirección y cuentan una historia
coherente: **la IA no pierde por lo que dice, sino por dónde termina la conversación.**

- **Cierra menos:** 25 % de compromisos firmes frente al 59 % de los humanos (−34 pp).
- **Se queda sin desenlace:** 43 % de sus llamadas terminan sin conclusión, frente al 4 %
  de las humanas. Es el hallazgo más grande de todo el análisis (+39 pp).
- **No aterriza el pago:** solo 20 % deja claro cómo, cuándo y dónde pagar, contra 59 %
  (−38 pp). Aun cuando la IA logra un acuerdo, el dinero se queda sin ruta.
- **No negocia ante la objeción:** cuando el deudor objeta, el humano responde con algo
  concreto (nueva fecha, nuevo monto) en 62 % de los casos; la IA en 29 % (−33 pp).
- **Genera desconfianza:** la objeción "no reconozco esta deuda / ¿quién me llama?" aparece
  en 27 % de las llamadas de IA contra 11 % de las humanas — la única no significativa al
  5 % (p = 0,06), así que se reporta como señal, no como conclusión.

El contraste de tono resultó al revés de lo que se temía al diseñar la variable: no es que el
LLM etiquete todo como "cordial" por defecto. Discrimina bien y muestra que la IA suena
neutra (52 %) mucho más que los humanos (9 %).
